# 01a · Base overview del dashboard de equipos

Resumen visual y tabular de los splits de resultados (Win/Loss) para entender rápidamente cómo varían las métricas clave de cada equipo.

## Objetivo del notebook

1. Cargar el parquet consolidado de splits generales y un ejemplo del origen crudo.
2. Confirmar que cada equipo tiene dos filas (victorias y derrotas).
3. Construir tablas resumen y rankings de métricas clave.
4. Generar visualizaciones útiles y guardar figuras/csv para reutilización.

## 1. Configuración y rutas

Importamos las librerías necesarias, definimos rutas y creamos carpetas de salida para tablas y gráficos.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Directorio base asumido como la raíz del repositorio
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / '00_data'

INTERMEDIATE_PATH = (
    DATA_DIR
    / '00b_intermediate'
    / 'team_dashboard'
    / 'general_splits'
    / '2024-25'
    / 'Regular Season'
    / 'team_dashboard__general_splits.parquet'
)
RAW_SAMPLE_PATH = (
    DATA_DIR
    / '00a_raw'
    / 'team_dashboard'
    / 'team_dashboard_by_general_splits'
    / '2024-25'
    / 'Regular Season'
    / 'team_dashboard_by_general_splits__1610612737__dataset_0.parquet'
)

TABLES_DIR = DATA_DIR / '00d_analysis' / 'base_overview'
FIGURES_DIR = BASE_DIR / '00e_reports' / 'figures' / 'base_overview'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

INTERMEDIATE_PATH, RAW_SAMPLE_PATH


## 2. Carga de datos

Leemos el parquet consolidado y un archivo crudo de referencia para confirmar la estructura de origen. Mostramos tamaños y un vistazo rápido.

In [ ]:
general_df = pd.read_parquet(INTERMEDIATE_PATH)
raw_sample_df = pd.read_parquet(RAW_SAMPLE_PATH)

print(f'Total filas (consolidado): {len(general_df):,}')
print(f'Columnas (consolidado): {general_df.shape[1]}')
print(f'Total filas (crudo ejemplo): {len(raw_sample_df):,}')

general_df.head()


In [ ]:
raw_sample_df.head()


## 3. Validación de splits Win/Loss

Filtramos el split `game_result` y comprobamos que cada equipo tiene exactamente dos filas (victoria y derrota). Reportamos cualquier anomalía.

In [ ]:
wl_df = general_df.query('split_type == \"game_result\"').copy()

team_split_counts = (
    wl_df.groupby(['TEAM_ID', 'TEAM_NAME'], dropna=False)['split_value'].nunique()
)
count_summary = team_split_counts.value_counts().sort_index()
print('Distribución de número de splits por equipo:')
print(count_summary)

teams_with_issues = team_split_counts[team_split_counts != 2]
if not teams_with_issues.empty:
    print('
Equipos con un número distinto a 2:')
    display(teams_with_issues)
else:
    print('
Todos los equipos tienen exactamente dos filas (W y L).')


## 4. Tablas resumen por equipo

Construimos una tabla con valores en victorias (`_W`), derrotas (`_L`) y la diferencia (`_DIFF`) para las métricas clave solicitadas.

In [ ]:
metrics = ['W_PCT', 'NET_RATING', 'OFF_RATING', 'DEF_RATING', 'PACE', 'EFG_PCT', 'TOV_PCT', 'OREB_PCT', 'FTR']

wins = (
    wl_df[wl_df['split_value'] == 'W']
    .set_index(['TEAM_ID', 'TEAM_NAME'])
    [['W'] + metrics]
    .rename(columns={col: f'{col}_W' for col in ['W'] + metrics})
)
losses = (
    wl_df[wl_df['split_value'] == 'L']
    .set_index(['TEAM_ID', 'TEAM_NAME'])
    [['L'] + metrics]
    .rename(columns={col: f'{col}_L' for col in ['L'] + metrics})
)
summary = wins.join(losses, how='inner').reset_index()
summary['WL_DIFF'] = summary['W_W'] - summary['L_L']
for metric in metrics:
    summary[f'{metric}_DIFF'] = summary[f'{metric}_W'] - summary[f'{metric}_L']

summary.head()


Guardamos la tabla resumen principal para poder reutilizarla fuera del notebook.

In [ ]:
summary_path = TABLES_DIR / 'team_win_loss_summary.csv'
summary.to_csv(summary_path, index=False)
summary_path


## 5. Rankings clave

Generamos rankings por diferencia de `NET_RATING`, por diferencial de victorias (`WL_DIFF`) y para los Four Factors (`EFG_PCT`, `TOV_PCT`, `OREB_PCT`, `FTR`).

In [ ]:
net_rating_rankings = summary.sort_values('NET_RATING_DIFF', ascending=False)[
    ['TEAM_ID', 'TEAM_NAME', 'NET_RATING_W', 'NET_RATING_L', 'NET_RATING_DIFF']
]
wl_diff_rankings = summary.sort_values('WL_DIFF', ascending=False)[
    ['TEAM_ID', 'TEAM_NAME', 'W_W', 'L_L', 'WL_DIFF']
]
four_factors = ['EFG_PCT', 'TOV_PCT', 'OREB_PCT', 'FTR']
four_factor_rankings = (
    summary.melt(
        id_vars=['TEAM_ID', 'TEAM_NAME'],
        value_vars=[f'{metric}_DIFF' for metric in four_factors],
        var_name='METRIC',
        value_name='DIFF'
    )
)
four_factor_rankings['METRIC'] = four_factor_rankings['METRIC'].str.replace('_DIFF', '', regex=False)
four_factor_rankings['RANK'] = four_factor_rankings.groupby('METRIC')['DIFF'].rank(ascending=False, method='min')

net_rating_rankings.head()


In [ ]:
wl_diff_rankings.head()


In [ ]:
four_factor_rankings.head()


Guardamos los rankings en CSV para análisis posteriores.

In [ ]:
net_rating_rankings_path = TABLES_DIR / 'ranking_net_rating.csv'
wl_diff_rankings_path = TABLES_DIR / 'ranking_wl_diff.csv'
four_factor_rankings_path = TABLES_DIR / 'ranking_four_factors.csv'

net_rating_rankings.to_csv(net_rating_rankings_path, index=False)
wl_diff_rankings.to_csv(wl_diff_rankings_path, index=False)
four_factor_rankings.to_csv(four_factor_rankings_path, index=False)

net_rating_rankings_path, wl_diff_rankings_path, four_factor_rankings_path


## 6. Visualizaciones

Creamos y guardamos los gráficos solicitados en la carpeta de reportes.

### 6.1 Barra de ΔNET_RATING por equipo

Ordenamos por el diferencial para identificar rápidamente qué equipos cambian más entre victorias y derrotas.

In [ ]:
bar_data = summary.sort_values('NET_RATING_DIFF', ascending=False)
fig, ax = plt.subplots(figsize=(10, 12))
sns.barplot(data=bar_data, x='NET_RATING_DIFF', y='TEAM_NAME', palette='coolwarm', ax=ax)
ax.axvline(0, color='black', linewidth=1, linestyle='--')
ax.set_title('Diferencial de Net Rating (Victorias - Derrotas)')
ax.set_xlabel('Δ NET_RATING')
ax.set_ylabel('Equipo')
fig.tight_layout()
delta_net_rating_path = FIGURES_DIR / 'delta_net_rating.png'
fig.savefig(delta_net_rating_path, dpi=300, bbox_inches='tight')
plt.close(fig)
delta_net_rating_path


### 6.2 Comparativa Four Factors (W vs L)

Usamos diagramas de dispersión Wins vs Losses para cada factor, con la línea identidad como referencia.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharex=False, sharey=False)
axes = axes.ravel()
for ax, metric in zip(axes, four_factors):
    x = summary[f'{metric}_L']
    y = summary[f'{metric}_W']
    sns.scatterplot(x=x, y=y, ax=ax)
    lims = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(lims, lims, linestyle='--', color='gray', linewidth=1)
    ax.set_xlabel(f'{metric} en derrotas')
    ax.set_ylabel(f'{metric} en victorias')
    ax.set_title(metric)
fig.suptitle('Comparativa Four Factors (W vs L)')
fig.tight_layout(rect=[0, 0.03, 1, 0.97])
four_factors_path = FIGURES_DIR / 'four_factors_comparison.png'
fig.savefig(four_factors_path, dpi=300, bbox_inches='tight')
plt.close(fig)
four_factors_path


### 6.3 Dispersión PACE vs OFF_RATING (W/L)

Graficamos cada equipo dos veces (victoria y derrota) para ver cómo cambia el ritmo ofensivo según el resultado.

In [ ]:
pace_off_records = []
for _, row in summary.iterrows():
    pace_off_records.append({
        'TEAM_NAME': row['TEAM_NAME'],
        'RESULT': 'Win',
        'PACE': row['PACE_W'],
        'OFF_RATING': row['OFF_RATING_W']
    })
    pace_off_records.append({
        'TEAM_NAME': row['TEAM_NAME'],
        'RESULT': 'Loss',
        'PACE': row['PACE_L'],
        'OFF_RATING': row['OFF_RATING_L']
    })
pace_off_df = pd.DataFrame(pace_off_records)
fig, ax = plt.subplots(figsize=(10, 8))
sns.scatterplot(data=pace_off_df, x='PACE', y='OFF_RATING', hue='RESULT', style='RESULT', ax=ax)
ax.set_title('PACE vs OFF_RATING por resultado')
ax.set_xlabel('PACE')
ax.set_ylabel('OFF_RATING')
fig.tight_layout()
pace_off_path = FIGURES_DIR / 'pace_vs_off_rating.png'
fig.savefig(pace_off_path, dpi=300, bbox_inches='tight')
plt.close(fig)
pace_off_path


### 6.4 Distribución de NET_RATING

Observamos la distribución de `NET_RATING` en victorias y derrotas para detectar diferencias en la dispersión.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(
    data=wl_df,
    x='NET_RATING',
    hue='split_value',
    element='step',
    stat='density',
    common_norm=False,
    ax=ax
)
ax.set_title('Distribución de Net Rating (W vs L)')
ax.set_xlabel('NET_RATING')
ax.set_ylabel('Densidad')
fig.tight_layout()
net_rating_dist_path = FIGURES_DIR / 'net_rating_distribution.png'
fig.savefig(net_rating_dist_path, dpi=300, bbox_inches='tight')
plt.close(fig)
net_rating_dist_path


### 6.5 Heatmap de correlaciones

Calculamos la correlación entre los diferenciales (`_DIFF`) para ver qué métricas se mueven en conjunto.

In [ ]:
corr_columns = ['WL_DIFF'] + [f'{metric}_DIFF' for metric in metrics]
corr_matrix = summary[corr_columns].corr()
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlación entre diferenciales (W - L)')
fig.tight_layout()
corr_heatmap_path = FIGURES_DIR / 'differentials_correlation_heatmap.png'
fig.savefig(corr_heatmap_path, dpi=300, bbox_inches='tight')
plt.close(fig)
corr_heatmap_path


## 7. Próximos pasos

- Incorporar anotaciones adicionales (por ejemplo, abreviaturas de equipos) en los gráficos para mejorar la lectura.
- Integrar estos resultados con notebooks posteriores para profundizar en segmentos específicos (local/visita, descansos, etc.).